# P02a Notebook 2: 数据清洗与存储

**任务**：清洗原始数据（缺失值、日期格式、数据类型、去重、极端收益率标记）、宽长转换、多频率合并，存储为 CSV + SQLite。

In [1]:
import os
import glob
import sqlite3
import pandas as pd
import numpy as np

RAW_STOCK  = '../data/raw/stocks'
RAW_INDEX  = '../data/raw/index'
RAW_MACRO  = '../data/raw/macro'
RAW_FIN    = '../data/raw/financial'
PROCESSED  = '../data/processed'
DB_PATH    = '../data/processed/findata.db'

os.makedirs(PROCESSED, exist_ok=True)
print('路径配置完成')

路径配置完成


## 2.1 加载并清洗股票日度数据

In [2]:
def load_and_clean_stock(filepath):
    """加载单只股票CSV，执行六项清洗任务"""
    code = os.path.basename(filepath).replace('.csv', '')
    df = pd.read_csv(filepath)

    # ① 标准化日期格式
    date_col = [c for c in df.columns if '日期' in c or 'date' in c.lower()][0]
    df.rename(columns={date_col: 'date'}, inplace=True)
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

    # ② 数据类型转换：确保OHLCV为float/int
    col_map = {}
    for c in df.columns:
        if any(k in c for k in ['开盘','收盘','最高','最低','涨跌幅','成交量','成交额',
                                 'open','high','low','close','volume']):
            col_map[c] = c
            df[c] = pd.to_numeric(df[c], errors='coerce')

    # ③ 检测并处理缺失值
    missing_before = df.isnull().sum().sum()
    df.dropna(subset=['date'], inplace=True)  # 日期缺失直接丢弃
    df.sort_values('date', inplace=True)
    df.ffill(inplace=True)                    # 价格缺失前向填充
    missing_after = df.isnull().sum().sum()

    # ④ 去除重复行
    dup_count = df.duplicated(subset=['date']).sum()
    df.drop_duplicates(subset=['date'], keep='last', inplace=True)

    # ⑤ 标识极端收益率（日涨跌幅超±20%）
    # 找收盘价列
    close_col = [c for c in df.columns if '收盘' in c or c.lower() == 'close'][0]
    df['daily_return'] = df[close_col].pct_change()
    df['is_extreme'] = df['daily_return'].abs() > 0.20

    df['code'] = code
    df.reset_index(drop=True, inplace=True)

    print(f'{code}: 缺失值 {missing_before}→{missing_after}, 重复行 {dup_count}, '
          f'极端收益 {df["is_extreme"].sum()} 条, 共{len(df)}行')
    return df


stock_files = glob.glob(f'{RAW_STOCK}/*.csv')
if not stock_files:
    print('警告：未找到股票原始文件，请先运行 Notebook 1')
else:
    stock_dfs = [load_and_clean_stock(f) for f in sorted(stock_files)]
    stocks_long = pd.concat(stock_dfs, ignore_index=True)
    print(f'\n股票长格式合并：{stocks_long.shape}')

000625: 缺失值 0→0, 重复行 0, 极端收益 0 条, 共1212行
000858: 缺失值 0→0, 重复行 0, 极端收益 0 条, 共1212行
600028: 缺失值 0→0, 重复行 0, 极端收益 0 条, 共1212行
600036: 缺失值 0→0, 重复行 0, 极端收益 0 条, 共1212行
600276: 缺失值 0→0, 重复行 0, 极端收益 0 条, 共1212行
600519: 缺失值 0→0, 重复行 0, 极端收益 0 条, 共1212行
600887: 缺失值 0→0, 重复行 0, 极端收益 0 条, 共1212行
600941: 缺失值 0→0, 重复行 0, 极端收益 0 条, 共725行
601088: 缺失值 0→0, 重复行 0, 极端收益 0 条, 共1212行
601398: 缺失值 0→0, 重复行 0, 极端收益 0 条, 共1212行

股票长格式合并：(11633, 10)


## 2.2 宽格式转换（Wide format — 每股一列收盘价）

In [3]:
if stock_files:
    # 找收盘价列名（可能因akshare版本而异）
    close_col_name = [c for c in stocks_long.columns
                      if '收盘' in c or c.lower() == 'close'][0]

    stocks_wide = stocks_long.pivot_table(
        index='date', columns='code', values=close_col_name
    )
    stocks_wide.columns.name = None
    stocks_wide.reset_index(inplace=True)
    print('宽格式（收盘价）：', stocks_wide.shape)
    stocks_wide.head(3)

宽格式（收盘价）： (1212, 11)


## 2.3 清洗指数数据

In [4]:
def load_index(code):
    fp = f'{RAW_INDEX}/{code}.csv'
    if not os.path.exists(fp):
        print(f'指数文件不存在: {fp}')
        return None
    df = pd.read_csv(fp)
    df.columns = df.columns.str.lower()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df.dropna(subset=['date'], inplace=True)
    df.sort_values('date', inplace=True)
    df.drop_duplicates(subset=['date'], keep='last', inplace=True)
    for c in ['open','high','low','close','volume']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    df['code'] = code
    print(f'指数 {code}: {df.shape}')
    return df

hs300 = load_index('000300')
sh    = load_index('000001')

指数 000300: (1212, 7)
指数 000001: (1212, 7)


## 2.4 清洗宏观数据

In [5]:
def clean_macro(filepath, value_col_hint):
    """清洗宏观数据；CPI文件自动过滤全国总指数行"""
    if not os.path.exists(filepath):
        print(f'宏观文件不存在: {filepath}')
        return None
    df = pd.read_csv(filepath)
    print(f'宏观原始列名: {list(df.columns)}')

    # ── CPI特殊处理：过滤全国行，取今值列 ─────────────────────
    if '商品' in df.columns:
        # 保留全国CPI一行/月
        mask = df['商品'].str.contains('全国', na=False)
        if mask.any():
            df = df[mask].copy()
        # 保留日期和今值两列
        keep = [c for c in ['日期', '今值'] if c in df.columns]
        df = df[keep].copy()
        df.rename(columns={'今值': 'cpi'}, inplace=True)

    # ── 寻找日期列 ─────────────────────────────────────────
    date_candidates = [c for c in df.columns if any(k in c for k in
                       ['日期', '月份', '年份', 'date', 'month', 'year'])]
    if not date_candidates:
        date_candidates = [df.columns[0]]
    df.rename(columns={date_candidates[0]: 'date'}, inplace=True)

    # ── 日期格式解析（兼容多种格式）────────────────────────────
    raw_dates = df['date'].astype(str).str.strip()
    parsed = None
    for fmt in ('%Y年%m月', '%Y-%m', '%Y%m', '%Y/%m', '%Y年%m月%d日', '%Y-%m-%d'):
        try:
            parsed = pd.to_datetime(raw_dates, format=fmt, errors='raise')
            print(f'  日期格式识别为: {fmt}')
            break
        except Exception:
            continue
    if parsed is None:
        parsed = pd.to_datetime(raw_dates, errors='coerce')
        print('  日期格式：自动推断')
    df['date'] = parsed
    df.dropna(subset=['date'], inplace=True)
    df.sort_values('date', inplace=True)

    # ── 数值列转换 ─────────────────────────────────────────
    for c in df.columns:
        if c != 'date':
            df[c] = pd.to_numeric(df[c], errors='coerce')
    df.ffill(inplace=True)
    print(f'  清洗后：{df.shape}')
    return df

cpi = clean_macro('../data/raw/macro/cpi_monthly.csv', 'cpi')
m2  = clean_macro('../data/raw/macro/m2_monthly.csv',  'm2')

宏观原始列名: ['商品', '日期', '今值', '预测值', '前值']
  日期格式识别为: %Y-%m-%d
  清洗后：(357, 5)
宏观原始列名: ['月份', '货币和准货币(M2)-数量(亿元)', '货币和准货币(M2)-同比增长', '货币和准货币(M2)-环比增长', '货币(M1)-数量(亿元)', '货币(M1)-同比增长', '货币(M1)-环比增长', '流通中的现金(M0)-数量(亿元)', '流通中的现金(M0)-同比增长', '流通中的现金(M0)-环比增长']
  日期格式：自动推断
  清洗后：(0, 10)


/var/folders/lw/ftxd8xhn3w5b51g4pxy6sjyr0000gt/T/ipykernel_42955/4270154870.py:27: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(raw_dates, errors='coerce')


## 2.4b 清洗财务指标数据（长格式验证）

In [ ]:
# ── 加载并清洗财务指标（长格式）────────────────────────────────
fin_path = f'{RAW_FIN}/financial_indicators.csv'
if os.path.exists(fin_path):
    fin = pd.read_csv(fin_path)
    print(f'财务数据原始列名: {list(fin.columns)}')
    print(f'原始形状: {fin.shape}')

    # 确保 value 列为数值型
    if 'value' in fin.columns:
        fin['value'] = pd.to_numeric(fin['value'], errors='coerce')

    # 去除空值
    fin.dropna(subset=['value'], inplace=True)

    # 只保留关键指标（ROE/ROA/净利率等）
    key_indicators = ['roeAvg', 'npMargin', 'gpMargin', 'netProfit', 'eps']
    if 'indicator' in fin.columns:
        fin_key = fin[fin['indicator'].isin(key_indicators)].copy()
        if fin_key.empty:
            fin_key = fin.copy()  # 如果没有匹配，保留全部
    else:
        fin_key = fin.copy()

    fin_key.to_csv(f'{PROCESSED}/financial_long.csv', index=False, encoding='utf-8-sig')
    print(f'财务数据（长格式）清洗后: {fin_key.shape}')
    print(fin_key.head())
else:
    print(f'财务文件不存在: {fin_path}，跳过')
    fin_key = None

## 2.4c 宽格式 ↔ 长格式转换演示

In [ ]:
# ── 宽格式 → 长格式（melt演示）────────────────────────────────
if 'stocks_wide' in dir() and stocks_wide is not None:
    # 宽→长
    stocks_long_demo = stocks_wide.melt(
        id_vars=['date'],
        var_name='code',
        value_name='close',
    )
    stocks_long_demo.dropna(subset=['close'], inplace=True)
    print(f'宽→长转换结果: {stocks_long_demo.shape}')
    print(stocks_long_demo.head())

    # 长→宽（pivot验证）
    stocks_wide_check = stocks_long_demo.pivot_table(
        index='date', columns='code', values='close'
    )
    stocks_wide_check.columns.name = None
    print(f'\n长→宽还原验证: {stocks_wide_check.shape}')


## 2.5 多频率数据合并（日度股票 + 月度宏观）

In [6]:
if stock_files and cpi is not None:
    # 将月度宏观数据扩展到日频（前向填充）
    cpi_value_col = [c for c in cpi.columns if c != 'date'][0]
    cpi_daily = (
        stocks_wide[['date']]
        .merge(cpi[['date', cpi_value_col]].rename(columns={'date': 'month'}),
               how='left',
               left_on=stocks_wide['date'].dt.to_period('M').dt.to_timestamp(),
               right_on='month')
        .drop(columns=['key_0', 'month'], errors='ignore')
    )

    # 简单方式：直接 merge_asof
    cpi_tmp = cpi[['date', cpi_value_col]].copy()
    cpi_tmp.columns = ['date', 'cpi']
    cpi_tmp['date'] = pd.to_datetime(cpi_tmp['date'])

    stocks_wide_sorted = stocks_wide.sort_values('date')
    merged = pd.merge_asof(
        stocks_wide_sorted,
        cpi_tmp.sort_values('date'),
        on='date',
        direction='backward',
    )
    print(f'合并后（含CPI）：{merged.shape}')
    merged.head(3)

合并后（含CPI）：(1212, 12)


## 2.6 存储：CSV（必选A）+ SQLite（进阶C）

In [7]:
# ── 方法A: CSV ──────────────────────────────────────────────
if stock_files:
    stocks_long.to_csv(f'{PROCESSED}/stocks_long.csv',
                       index=False, encoding='utf-8-sig')
    stocks_wide.to_csv(f'{PROCESSED}/stocks_wide.csv',
                       index=False, encoding='utf-8-sig')
    print('CSV 已保存: stocks_long.csv, stocks_wide.csv')

if hs300 is not None:
    hs300.to_csv(f'{PROCESSED}/hs300.csv', index=False, encoding='utf-8-sig')
    print('CSV 已保存: hs300.csv')

if cpi is not None:
    cpi.to_csv(f'{PROCESSED}/cpi.csv', index=False, encoding='utf-8-sig')

if m2 is not None:
    m2.to_csv(f'{PROCESSED}/m2.csv', index=False, encoding='utf-8-sig')

print('所有 CSV 保存完毕')

CSV 已保存: stocks_long.csv, stocks_wide.csv
CSV 已保存: hs300.csv
所有 CSV 保存完毕


In [8]:
# ── 方法C: SQLite ─────────────────────────────────────────────
import time

conn = sqlite3.connect(DB_PATH)

tables_saved = []
if stock_files:
    t0 = time.time()
    stocks_long_db = stocks_long.copy()
    stocks_long_db['date'] = stocks_long_db['date'].astype(str)
    stocks_long_db.to_sql('stocks_long', conn, if_exists='replace', index=False)
    t_csv_long = time.time() - t0

    t0 = time.time()
    sw = stocks_wide.copy()
    sw['date'] = sw['date'].astype(str)
    sw.to_sql('stocks_wide', conn, if_exists='replace', index=False)
    t_csv_wide = time.time() - t0

    tables_saved += ['stocks_long', 'stocks_wide']

if hs300 is not None:
    idx = hs300.copy()
    idx['date'] = idx['date'].astype(str)
    idx.to_sql('hs300', conn, if_exists='replace', index=False)
    tables_saved.append('hs300')

if cpi is not None:
    c = cpi.copy(); c['date'] = c['date'].astype(str)
    c.to_sql('cpi', conn, if_exists='replace', index=False)
    tables_saved.append('cpi')

if m2 is not None:
    m = m2.copy(); m['date'] = m['date'].astype(str)
    m.to_sql('m2', conn, if_exists='replace', index=False)
    tables_saved.append('m2')

conn.close()
print(f'SQLite 已保存：{DB_PATH}')
print(f'数据表：{tables_saved}')

SQLite 已保存：../data/processed/findata.db
数据表：['stocks_long', 'stocks_wide', 'hs300', 'cpi', 'm2']


In [9]:
# ── 性能对比：CSV vs SQLite 读取速度 ─────────────────────────
if stock_files:
    import time

    t0 = time.time()
    _ = pd.read_csv(f'{PROCESSED}/stocks_long.csv')
    t_csv = time.time() - t0

    t0 = time.time()
    conn = sqlite3.connect(DB_PATH)
    _ = pd.read_sql('SELECT * FROM stocks_long', conn)
    conn.close()
    t_sql = time.time() - t0

    print(f'读取性能对比：')
    print(f'  CSV    读取耗时: {t_csv:.3f}s')
    print(f'  SQLite 读取耗时: {t_sql:.3f}s')
    print(f'  说明：CSV适合直接查看，SQLite支持SQL查询且对大数据集更高效。')

读取性能对比：
  CSV    读取耗时: 0.017s
  SQLite 读取耗时: 0.012s
  说明：CSV适合直接查看，SQLite支持SQL查询且对大数据集更高效。
